In [1]:
import earthaccess
import xarray as xr

In [2]:
results = earthaccess.search_datasets(instrument="oci")
for item in results:
    summary = item.summary()
    print(summary["short-name"])

PACE_OCI_L0_SCI
PACE_OCI_L1A_SCI
PACE_OCI_L1B_SCI
PACE_OCI_L1C_SCI
PACE_OCI_L2_UVAI_UAA_NRT
PACE_OCI_L2_UVAI_UAA
PACE_OCI_L2_AER_UAA_NRT
PACE_OCI_L2_AER_UAA
PACE_OCI_L2_AOP_NRT
PACE_OCI_L2_AOP
PACE_OCI_L2_CLOUD_MASK_NRT
PACE_OCI_L2_CLOUD_MASK
PACE_OCI_L2_CLOUD_NRT
PACE_OCI_L2_CLOUD
PACE_OCI_L2_IOP_NRT
PACE_OCI_L2_IOP
PACE_OCI_L2_LANDVI_NRT
PACE_OCI_L2_LANDVI
PACE_OCI_L2_BGC
PACE_OCI_L2_BGC_NRT
PACE_OCI_L2_PAR_NRT
PACE_OCI_L2_PAR
PACE_OCI_L2_SFREFL_NRT
PACE_OCI_L2_SFREFL
PACE_OCI_L3B_AOT_NRT
PACE_OCI_L3B_AOT
PACE_OCI_L3B_AVW_NRT
PACE_OCI_L3B_AVW
PACE_OCI_L3B_CARBON
PACE_OCI_L3B_CARBON_NRT
PACE_OCI_L3B_CHL_NRT
PACE_OCI_L3B_CHL
PACE_OCI_L3B_KD_NRT
PACE_OCI_L3B_KD
PACE_OCI_L3B_FLH_NRT
PACE_OCI_L3B_FLH
PACE_OCI_L3B_IOP_NRT
PACE_OCI_L3B_IOP
PACE_OCI_L3B_LANDVI_NRT
PACE_OCI_L3B_LANDVI
PACE_OCI_L3B_PIC_NRT
PACE_OCI_L3B_PIC
PACE_OCI_L3B_POC_NRT
PACE_OCI_L3B_POC
PACE_OCI_L3B_PAR_NRT
PACE_OCI_L3B_PAR
PACE_OCI_L3B_RRS_NRT
PACE_OCI_L3B_RRS
PACE_OCI_L3B_SFREFL_NRT
PACE_OCI_L3B_SFREFL
PACE_OCI_L3M_UV

In [3]:
lon_min, lon_max = -98, -78
lat_min, lat_max = 18, 30.5
date_ini, date_end= '2025-05-02', '2025-05-13'
# Level 2 data
results = earthaccess.search_data(
    short_name = 'PACE_OCI_L2_AOP',
    temporal = (date_ini, date_end),
    bounding_box = (lon_min, lat_min, lon_max, lat_max)
)
len(results)

33

In [4]:
lon_min, lon_max = -98, -78
lat_min, lat_max = 18, 30.5
date_ini, date_end= '2024-03-22', '2024-04-09'
# Level 2 data
results = earthaccess.search_data(
    short_name = 'PACE_OCI_L2_AOP',
    temporal = (date_ini, date_end),
    bounding_box = (lon_min, lat_min, lon_max, lat_max)
)
len(results)
fileset = earthaccess.open(results);

QUEUEING TASKS | :   0%|          | 0/45 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/45 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/45 [00:00<?, ?it/s]

In [ ]:
# ---- Load Libraries ----
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import cartopy
import cartopy.crs as ccrs
from matplotlib import colors
from scipy.integrate import trapezoid
from scipy.spatial import cKDTree
from matplotlib.colors import Normalize

In [ ]:
for file_ in fileset[::5]:
    print(file_.full_name)
    
    # --- Extracción de la fecha ---
    raw_date = file_.full_name.split('.')[1].split('T')[0]
    hour_    = file_.full_name.split('.')[1].split('T')[1]
    formatted_date = f"{raw_date[:4]}-{raw_date[4:6]}-{raw_date[6:]} {hour_[0:2]}:{hour_[2:4]}"
    # ------------------------------

    datatree = xr.open_datatree(file_, decode_timedelta=False, chunks={}) 
    
    ds = xr.merge(datatree.to_dict().values())
    ds = ds.set_coords(("longitude", "latitude"))
    
    avw_da = ds["avw"]
    
    vmin = float(avw_da.quantile(0.01))
    vmax = float(avw_da.quantile(0.99))
    
    fig, ax = plt.subplots(figsize=(12, 6), subplot_kw={"projection": ccrs.PlateCarree()})
    
    ax.gridlines(draw_labels=True)
    ax.coastlines()
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    
    img = avw_da.plot(x="longitude", y="latitude", cmap="jet", vmin=vmin, vmax=vmax, ax=ax, cbar_kwargs={"pad": 0.1})
    
    ax.set_title(f"PACE OCI - Apparent Visible Wavelength (AVW)\nDate: {formatted_date}", fontsize=14, pad=20)
    
    plt.show()
    plt.close()

In [ ]:
file_.full_name.split('.')[1].split('T')

In [ ]:
ds.wavelength.values

In [ ]:
for file_ in fileset[::2]:
    print(file_.full_name)
    
    # --- Extracción de la fecha ---
    raw_date = file_.full_name.split('.')[1].split('T')[0]
    hour_    = file_.full_name.split('.')[1].split('T')[1]
    formatted_date = f"{raw_date[:4]}-{raw_date[4:6]}-{raw_date[6:]} {hour_[0:2]}:{hour_[2:4]}"
    # ------------------------------

    datatree = xr.open_datatree(file_, decode_timedelta=False, chunks={}) 
    # 1. Filter wavelengths > 700 nm
    
    ds = xr.merge(datatree.to_dict().values())
    ds = ds.set_coords(("longitude", "latitude"))
    
    rrs_da = ds["Rrs"].sel(wavelength_3d=900,method="nearest")
    
    vmin = float(rrs_da.quantile(0.01))
    vmax = float(rrs_da.quantile(0.99))
    
    fig, ax = plt.subplots(figsize=(12, 6), subplot_kw={"projection": ccrs.PlateCarree()})
    
    ax.gridlines(draw_labels=True)
    ax.coastlines()
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    
    img = rrs_da.plot(x="longitude", y="latitude", cmap="jet", vmin=vmin, vmax=vmax, ax=ax, cbar_kwargs={"pad": 0.1})
    
    ax.set_title(f"PACE OCI - Remote sensing reflectance (RRS) 900nm \nDate: {formatted_date}", fontsize=14, pad=20)
    
    plt.show()
    plt.close()

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import earthaccess
import numpy as np

# --- 1. Configuración de parámetros ---
lon_min, lon_max = -98, -78
lat_min, lat_max = 18, 30.5
date_ini, date_end= '2025-05-02', '2025-05-13'
# Level 2 data
results = earthaccess.search_data(
    short_name = 'PACE_OCI_L2_AOP',
    temporal = (date_ini, date_end),
    bounding_box = (lon_min, lat_min, lon_max, lat_max)
)
len(results)
fileset = earthaccess.open(results);

print(f"Archivos encontrados: {len(results)}")

# --- 3. Procesamiento y Visualización ---
# Usamos un subconjunto para pruebas (fileset[::5])
for file_ in results[::5]:
    try:
        print(f"\nProcesando: {file_.full_name}")
        
        # Extracción de fecha para el título
        # Estructura esperada: ...PACE_OCI.20250513T172205...
        raw_date = file_.full_name.split('.')[1].split('T')[0]
        formatted_date = f"{raw_date[:4]}-{raw_date[4:6]}-{raw_date[6:]}"

        # Abrir datatree (necesario para grupos en PACE L2)
        # Usamos fsspec para abrir directamente desde S3 si es posible
        with earthaccess.open([file_])[0] as f:
            dt = xr.open_datatree(f, decode_timedelta=False, chunks={})
            
            # Combinar grupos: navigation_data (lat/lon) y geophysical_data (avw, etc)
            ds = xr.merge(dt.to_dict().values())
            
            # Establecer coordenadas explícitas para datos de barrido (Swath)
            # Esto resuelve el error de "Dimensions not exist"
            ds = ds.set_coords(("longitude", "latitude"))
            
            # Seleccionar variable
            avw_da = ds["avw"]

            # --- Filtro de área (Transecto) ---
            # Verificamos si hay píxeles con coordenadas dentro de nuestro cuadro
            mask = (
                (ds.latitude >= lat_min) & (ds.latitude <= lat_max) & 
                (ds.longitude >= lon_min) & (ds.longitude <= lon_max)
            )

            if not mask.any():
                print(f"--> Saltando: No hay datos dentro de la BBOX para esta fecha.")
                continue

            # Recortar datos para acelerar el renderizado y evitar outliers globales
            subset = avw_da.where(mask, drop=True)
            
            # Calcular límites de color dinámicos
            vmin = float(subset.quantile(0.01))
            vmax = float(subset.quantile(0.99))

            # --- Configuración del Mapa ---
            fig = plt.figure(figsize=(12, 7))
            ax = plt.axes(projection=ccrs.PlateCarree())
            
            ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
            ax.coastlines(resolution='10m', color='black', linewidth=1)
            gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
            gl.top_labels = False
            gl.right_labels = False

            # Graficar usando las coordenadas de longitud y latitud del sensor
            img = subset.plot(
                x="longitude", 
                y="latitude", 
                ax=ax, 
                cmap="jet", 
                vmin=vmin, 
                vmax=vmax,
                cbar_kwargs={"pad": 0.05, "label": "AVW (nm)"},
                transform=ccrs.PlateCarree()
            )

            ax.set_title(f"PACE OCI - Apparent Visible Wavelength (AVW)\nFecha: {formatted_date}", 
                         fontsize=14, pad=20)
            
            plt.show()
            plt.close()

    except Exception as e:
        print(f"Error al procesar")

In [ ]:
    ds = xr.merge(datatree.to_dict().values())
    ds = ds.set_coords(("longitude", "latitude"))

In [ ]:
ds.wavelength_3d